
**TAISS 2026 - Filière F1 (Data Science)**

**TP2 -  Segmentation RFM**

**Partie 1 :** Construction et transformation des features RFM

**Entrée :** `data/processed/transactions_clean.parquet` (Partie 1).

**Sortie :** une ligne par client, avec les features RFM brutes et transformées, prêtes pour le clustering de la partie 3. 

Ce notebook répond directement aux **questions 2 et 3 du rapport** : <br>
- Q2 : pourquoi ne pas appliquer K-means sur les RFM bruts ?
- Q3 : Fréquence et Montant sont corrélés : comment traiter cette redondance ?



In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Configuration de l'affichage Pandas et du style des graphiques
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
sns.set_theme(style="whitegrid")

# 2. Définition des dossiers de données et de figures
PROC = Path("../data/processed")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)

# 3. Chargement des données nettoyées sauvegardées en Parquet
df = pd.read_parquet(PROC / "transactions_clean.parquet")

# 4. Affichage du bilan de chargement et aperçu des premières lignes
print(f"{len(df):,} transactions | {df.CustomerID.nunique():,} clients uniques")
display(df.head(3))

**2.1 Date de référence :** <br>
La Récence (R) se mesure par rapport à une date de référence. Le dataset s'arrête au 09/12/2011. On prend **le lendemain de la dernière transaction** comme snapshot.


In [ ]:
# 1. Définition de la date de référence (SNAPSHOT) pour le calcul de la Récence
# On prend la date de la toute dernière transaction et on ajoute 1 jour de marge.
# Ainsi, un client ayant acheté le dernier jour aura une récence de 1 jour (et non 0).
SNAPSHOT = df["InvoiceDate"].max() + pd.Timedelta(days=1)

# 2. Affichage de la date maximale observée et de la date snapshot retenue
print("Dernière transaction :", df["InvoiceDate"].max())
print("Snapshot retenu      :", SNAPSHOT)

**2.2 Agrégation par client: RFM** <br>
**Récence (R) :** Jours depuis le dernier achat (plus c'est bas, mieux c'est) <br> 
**Fréquence (F) :** Nombre de **factures distinctes** <br>
**Montant (M) :** Somme des `Amount`, CA total généré sur la période. 
Ici On ajoutera deux variables de contrôle, non utilisées pour le clustering mais utiles à l'interprétation en partie 4 : `Anciennete` (jours depuis le **premier** achat) et`PanierMoyen` (Montant / Fréquence).

In [ ]:
# 1. Agrégation des transactions au niveau de chaque CustomerID
rfm = (
    df.groupby("CustomerID")
    .agg(
        Recence=(
            "InvoiceDate",
            lambda s: (SNAPSHOT - s.max()).days,
        ),  # Jours depuis le dernier achat
        Frequence=("Invoice", "nunique"),  # Nombre de commandes uniques
        Montant=("Amount", "sum"),  # Dépense totale cumulative
        Anciennete=(
            "InvoiceDate",
            lambda s: (SNAPSHOT - s.min()).days,
        ),  # Jours depuis la première commande
        NbArticles=("Quantity", "sum"),  # Volume total d'articles achetés
    )
    .reset_index()
)

# 2. Calcul du Panier Moyen par facture
rfm["PanierMoyen"] = rfm["Montant"] / rfm["Frequence"]

# 3. Validation des dimensions du DataFrame (devrait correspondre au nombre de clients uniques)
print("Nombre de clients et colonnes créées :", rfm.shape)

# 4. Aperçu des premières lignes de la table RFM augmentée
display(rfm.head())

In [ ]:
**Distribution des Variables**

In [ ]:
# Affichage des statistiques descriptives des variables RFM et des métriques dérivées
# Les percentiles 95% et 99% permettent de repérer l'asymétrie et la présence d'outliers
rfm[["Recence", "Frequence", "Montant", "Anciennete", "PanierMoyen"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
).round(1)

**Analyse de la Structure Client et Concentration du Chiffre d'Affaires**

In [ ]:
# 1. Proportion de clients 'mono-achat' (qui n'ont passé qu'une seule commande)
mono = (rfm["Frequence"] == 1).mean()

# 2. Part du Chiffre d'Affaires générée par les 1% de clients les plus importants
top1 = rfm["Montant"].nlargest(int(len(rfm) * 0.01)).sum() / rfm["Montant"].sum()

# 3. Part du Chiffre d'Affaires générée par les 10% de clients les plus importants (Loi de Pareto)
top10 = (
    rfm["Montant"].nlargest(int(len(rfm) * 0.10)).sum() / rfm["Montant"].sum()
)

# 4. Affichage des métriques clés de la base clients
print(f"Clients mono-achat         : {mono:.1%}")
print(f"Part du CA du top 1 %      : {top1:.1%}")
print(f"Part du CA du top 10 %     : {top10:.1%}")
print(f"Récence médiane            : {rfm['Recence'].median():.0f} jours")
print(
    f"Montant médian / moyen     : £{rfm['Montant'].median():,.0f} / £{rfm['Montant'].mean():,.0f}"
)

**Petit résumé** : environ 28 % des clients n'ont acheté qu'une fois, et le top 10 % des clients pèse environ 64 % du CA. L'écart entre le montant médian (856 £) et le montant moyen (2 917 £) signale une distribution très asymétrique, ce qui fait que **la moyenne n'est pas un résumé fiable ici**. 

---
**2.3 Distributions brutes**  <br>
Diagnostics d'Asymétrie et Justification de la Transformation Logarithmique

In [ ]:
# 1. Sélection des 3 variables de base de la segmentation RFM
RFM3 = ["Recence", "Frequence", "Montant"]

# 2. Calcul et affichage du coefficient d'asymétrie (skewness) sur les données brutes
# Un skewness > 1 indique une forte asymétrie à droite (longue traîne vers les grandes valeurs)
print("Skewness (données brutes) :")
print(rfm[RFM3].skew().round(2))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Création d'une grille de graphiques à 2 lignes et 3 colonnes (Histogrammes + Boxplots)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

# 2. Boucle d'itération sur chacune des variables RFM (Recence, Frequence, Montant)
for i, c in enumerate(RFM3):
    # Ligne 1 : Histogramme de la distribution brute avec indication du coefficient d'asymétrie (skewness)
    sns.histplot(rfm[c], bins=60, ax=axes[0, i], color="#4C72B0")
    axes[0, i].set_title(f"{c} — brut (skew={rfm[c].skew():.1f})")

    # Ligne 2 : Boîte à moustaches (Boxplot) pour observer les valeurs extrêmes (outliers)
    sns.boxplot(x=rfm[c], ax=axes[1, i], color="#4C72B0")

# 3. Ajustement de la disposition des sous-graphiques et sauvegarde de la figure
plt.tight_layout()
plt.savefig(FIG / "02_distributions_brutes.png", dpi=150)
plt.show()

**2.4 Question 2 du rapport : **  pourquoi pas K-means sur les RFM bruts ? <br>

**1. K-means minimise une distance euclidienne** ce qui fait que il est dominé par les grandes échelles. **Le Montant s'étale de 3 £ à 581 000 £, la Fréquence de 1 à 373, la Récence de 1 à 739. Sans mise à l'échelle, la distance entre deux clients est *presque entièrement* déterminée par le Montant, la Récence et la Fréquence ne pèsent quasiment rien. La segmentation « RFM » serait en réalité une segmentation « M ».

**2. Skewness extrême (Montant ≈ 25, Fréquence ≈ 12).** Une poignée de grossistes est si éloignée du nuage que K-means leur consacrerait des clusters d'une poignée d'individus, pendant que les 95 % de clients ordinaires seront écrasés dans un seul gros cluster indifférencié, donc inutilisable pour le marketing.


In [ ]:
**Transformation**

In [ ]:
import numpy as np
import pandas as pd

# 1. Application de la transformation logarithmique log1p = log(1 + x)
rfm_log = rfm[RFM3].apply(np.log1p)
rfm_log.columns = [f"log_{c}" for c in RFM3]

# 2. Tableau comparatif : impact sur le skewness et réduction de l'écart max/médiane
comp = pd.DataFrame(
    {
        "skew_brut": rfm[RFM3].skew().values,
        "skew_log": rfm_log.skew().values,
        "ratio_max/median_brut": (rfm[RFM3].max() / rfm[RFM3].median()).values,
    },
    index=RFM3,
).round(2)

# 3. Affichage du bilan comparatif
display(comp)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 1. Création d'une figure composée de 3 sous-graphiques alignés en ligne
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 2. Boucle d'itération sur les 3 variables RFM (Recence, Frequence, Montant)
for i, c in enumerate(RFM3):
    # Calcul de la transformation log1p et tracé de l'histogramme
    log_vals = np.log1p(rfm[c])
    sns.histplot(log_vals, bins=60, ax=axes[i], color="#55A868")

    # Affichage du coefficient d'asymétrie (skewness) révisé dans le titre
    axes[i].set_title(f"log1p({c}) — skew={log_vals.skew():.2f}")

# 3. Ajustement des marges, sauvegarde dans le dossier FIG et affichage
plt.tight_layout()
plt.savefig(FIG / "02_distributions_log.png", dpi=150)
plt.show()

> **Interpretation :** la transformation `log1p` fait passer la skewness du Montant de **25,3 à 0,27** et celle de la Fréquence de **12,0 à 1,00**, les distributions deviennent quasi-normales. On utilise `log1p` (= log(1+x)) plutôt que `log` par sécurité numérique, même si après nettoyage aucune valeur n'est nulle.

> **Pourquoi le log ne suffit pas et qu'il faut ensuite standardiser** : après log, les trois variables restent sur des plages différentes (log-Récence ~0–6,6 ; log-Montant ~1–13). La standardisation (moyenne 0, écart-type 1) leur donne **un poids égal** dans la distance euclidienne, ce qui est le choix explicite : on considère que R, F et M comptent autant l'un que l'autre.

**2.5 Standardisation**

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Instanciation et ajustement du StandardScaler sur les données log-transformées
scaler = StandardScaler()
X = scaler.fit_transform(rfm_log)

# 2. Conversion de la matrice NumPy résultante en DataFrame Pandas avec des noms de colonnes explicites
X = pd.DataFrame(
    X, columns=["R_scaled", "F_scaled", "M_scaled"], index=rfm.index
)

# 3. Vérification des propriétés fondamentales de la standardisation (moyenne ~ 0, écart-type ~ 1)
print("Moyennes     :", X.mean().round(6).values)
print("Écarts-types :", X.std().round(4).values)

# 4. Affichage du résumé statistique complet des variables centrées réduites
display(X.describe().round(2))

**Vérification visuelle : avant / après**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Création d'une figure à 2 sous-graphiques côte à côte pour comparer l'espace des données
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 2. Sous-graphique 1 : Nuage de points sur les données brutes (Frequence vs Montant)
# Illustre l'écrasement de la majorité des points près de l'origine à cause des valeurs extrêmes
sns.scatterplot(
    x=rfm.Frequence, y=rfm.Montant, s=8, alpha=0.4, ax=axes[0]
)
axes[0].set_title("Brut — quelques points écrasent tout le reste")

# 3. Sous-graphique 2 : Nuage de points après transformation log1p et StandardScaler
# Montre la révélation d'une structure continue et bien répartie, adaptée au clustering
sns.scatterplot(
    x=X.F_scaled, y=X.M_scaled, s=8, alpha=0.4, ax=axes[1], color="#55A868"
)
axes[1].set_title("log + standardisé — structure lisible")

# 4. Ajustement de la mise en page, sauvegarde de la figure et affichage
plt.tight_layout()
plt.savefig(FIG / "02_avant_apres_transformation.png", dpi=150)
plt.show()

**2.6 Question 3 du rapport** 

On mesure la corrélation **avant et après** transformation. 

In [ ]:
# 1. Matrice de corrélation de Pearson sur les données brutes
# Très sensible aux valeurs extrêmes et aux relations non linéaires
print("Pearson — brut :")
display(rfm[RFM3].corr().round(2))

# 2. Matrice de corrélation de Pearson après transformation log1p
# Mesure la relation linéaire sur l'échelle logarithmique
print("Pearson — log :")
display(rfm_log.corr().round(2))

# 3. Matrice de corrélation de Spearman (basée sur le rang des observations)
# Robuste aux outliers, elle reflète la véritable dépendance monotone initiale
print("Spearman (rangs, robuste aux outliers) :")
display(rfm[RFM3].corr(method="spearman").round(2))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Création d'une figure composée de 2 cartes de chaleur (Heatmaps) côte à côte
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# 2. Heatmap des corrélations de Pearson sur les données brutes
# Sensible aux outliers et aux relations non-linéaires
sns.heatmap(
    rfm[RFM3].corr(),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    ax=ax[0],
    fmt=".2f",
)
ax[0].set_title("Corrélations — brut")

# 3. Heatmap des corrélations de Pearson après transformation log1p
# Reflète les relations linéarisées prêtes pour les algorithmes à base de distance
sns.heatmap(
    rfm_log.corr(),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    ax=ax[1],
    fmt=".2f",
)
ax[1].set_title("Corrélations — log")

# 4. Ajustement des marges, sauvegarde dans le dossier FIG et affichage
plt.tight_layout()
plt.savefig(FIG / "02_correlations.png", dpi=150)
plt.show()

**Constat** : après log, corr(Fréquence, Montant) = **0,85** (Spearman 0,86). C'est très élevé (qui commande souvent dépense plus). La corrélation brute n'était que de 0,62, artificiellement basse parce que les outliers cassaient la relation linéaire.

**Conséquence sur K-means** : deux variables corrélées à 0,85 pointent presque dans la même direction. La distance euclidienne compte donc « l'intensité d'achat » **deux fois** et la Récence une seule donc l'axe R est implicitement sous-pondéré.

Quatre options possible pour gérer la forte corrélation entre la **Fréquence (F)** et le **Montant (M)** :

| Option | Description et effet | Décision et justification |
| :--- | :--- | :--- |
| **A. Ne rien faire (Garder R, F, M)** | $F$ et $M$ véhiculent une information partiellement redondante | **Retenu** — Socle principal pour une interprétation marketing directe. |
| **B. Supprimer le Montant (Garder R, F)** | Élimine complètement l'information de valeur | **Rejeté** — Le montant est indispensable pour analyser la valeur client (*LTV*). |
| **C. Remplacer M par le Panier Moyen ($M / F$)** | Décorrèle les variables mais modifie le sens métier | **Alternative** — Testé en annexe (mesure la taille de commande, pas la valeur totale). |
| **D. ACP (PCA) sur les 3 variables log-standardisées** | Décorrèle parfaitement par construction géométrique | **Alternative** — Testé ci-dessous à titre de diagnostic. |

---

**Justification du choix A (RFM Classique)**

1. **Interprétabilité et exploitation métier**
   * L'objectif principal de cette segmentation est d'obtenir des profils immédiatement lisibles par les équipes marketing et commerciales.
   * Les centroïdes calculés directement sur les variables $R, F, M$ s'expriment en métriques concrètes : *Quand le client est-il venu pour la dernière fois ? À quelle fréquence achetait-il ? Combien a-t-il dépensé au total ?*

2. **Compromis avec la Réduction de Dimensionnalité (ACP)**
   * Bien que l'ACP garantisse une indépendance linéaire totale entre les composantes, les centroïdes exprimés en axes polynomiaux abstraits nécessitent un aller-retour constant vers les variables initiales.
   * Conserver la lisibilité directe au prix d'une légère colinéarité $F/M$ apporte une valeur opérationnelle nettement supérieure pour la présentation des résultats.

3. **Redondance assumée et documentée**
   * La forte corrélation entre la Fréquence et le Montant est connue et intégrée. Elle aligne naturellement l'algorithme de clustering sur la dimension **« Valeur Client »**, ce qui répond précisément aux attentes stratégiques de la direction.

---

> **Vérification complémentaire :** Nous exécutons néanmoins l'ACP ci-dessous à titre de contrôle pour vérifier si la structure des clusters en serait fortement modifiée.

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA

# 1. Fit PCA model on standardized features (X)
pca = PCA().fit(X)

# 2. Print individual and cumulative explained variance ratios
print("Explained variance ratio :", pca.explained_variance_ratio_.round(3))
print("Cumulative variance      :", pca.explained_variance_ratio_.cumsum().round(3))

# 3. Display the component loadings (eigenvectors) matrix
loadings = pd.DataFrame(
    pca.components_,
    columns=RFM3,
    index=[f"PC{i+1}" for i in range(3)],
).round(2)

display(loadings)

> **Lecture** : PC1 capte **76 %** de la variance et charge positivement F et M, négativement R → c'est un axe **« valeur / engagement client »**. PC2 (19 %) est porté par la Récence. Les deux premières composantes couvrent 95 % de l'information : la structure est essentiellement **bidimensionnelle**, ce qui confirme la redondance F/M.

**Panier moyen au lieu du montant total**

In [ ]:
import numpy as np
import pandas as pd

# 1. Construction du jeu de données alternatif avec le Panier Moyen au lieu du Montant Total
alt = pd.DataFrame(
    {
        "log_R": np.log1p(rfm["Recence"]),
        "log_F": np.log1p(rfm["Frequence"]),
        "log_PanierMoyen": np.log1p(rfm["PanierMoyen"]),
    }
)

# 2. Calcul et affichage de la corrélation ciblée entre Fréquence et Panier Moyen
corr_f_pm = round(alt["log_F"].corr(alt["log_PanierMoyen"]), 2)
print("Corrélation F / PanierMoyen (log) :", corr_f_pm)

# 3. Affichage de la matrice de corrélation complète
display(alt.corr().round(2))

> Le panier moyen est bien moins corrélé à la Fréquence que le Montant total. Cette variante est conservée comme **test de robustesse** (Partie 3 / extension 3) : si les segments obtenus sont similaires, la conclusion est solide.

**2.7 Scores RFM par quintiles** 

In [ ]:
rfm["R_score"] = pd.qcut(rfm.Recence, 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm.Frequence.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = pd.qcut(rfm.Montant, 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_score"] = rfm.R_score.astype(str) + rfm.F_score.astype(str) + rfm.M_score.astype(str)
rfm["RFM_somme"] = rfm[["R_score","F_score","M_score"]].sum(axis=1)

print("Note : la Fréquence a beaucoup d'ex-aequo (28 % de clients à F=1),")
print("d'où le rank(method='first') pour forcer des quintiles de tailles égales.")
display(rfm.RFM_somme.value_counts().sort_index())
rfm[["CustomerID","Recence","Frequence","Montant","R_score","F_score","M_score","RFM_score"]].head()

**2.8 Sauvegarde** <br>
On sauvegarde trois objets :
- `rfm_features.parquet` : les features brutes + scores (pour l'interprétation, Partie 4) ;
- `rfm_scaled.parquet`   : la matrice log-standardisée (entrée du clustering, Partie 3) ;
- `scaler.joblib`        : le scaler ajusté, pour pouvoir **retransformer les centroïdes**  en unités métier en Partie 4 (`scaler.inverse_transform` puis `np.expm1`).

In [ ]:
import joblib

import joblib

X_full = X.copy()

# 1. Insertion de l'identifiant CustomerID en première position du DataFrame RFM
if "CustomerID" not in rfm.columns:
    rfm.insert(0, "CustomerID", rfm.CustomerID.values)

# 2. Sauvegarde des données agrégées et de la matrice standardisée au format Parquet
rfm.to_parquet(PROC / "rfm_features.parquet", index=False)
X_full.to_parquet(PROC / "rfm_scaled.parquet", index=False)

# 3. Export de l'objet StandardScaler et des métadonnées de prétraitement
joblib.dump(
    {"scaler": scaler, "snapshot": SNAPSHOT, "cols": RFM3},
    PROC / "scaler.joblib",
)

# 4. Confirmation de la sauvegarde et affichage des dimensions de la matrice
print(" Sauvegardé :", [p.name for p in PROC.glob("rfm*")] + ["scaler.joblib"])
print("Matrice de clustering :", X_full.shape)

**Mini-résumé** Les 776 582 transactions nettoyées sont agrégées en **5 852 clients**, décrits par la Récence (jours depuis le dernier achat), la Fréquence (nombre de **factures distinctes**, et non de lignes) et le Montant (CA cumulé), au snapshot du 10/12/2011, lendemain de la dernière transaction, retenu plutôt que la date du jour pour préserver le pouvoir discriminant de R. Les distributions brutes sont fortement asymétriques (skew de 25,3 pour le Montant, 12,0 pour> la Fréquence) et d'échelles incomparables : appliquer K-means directement reviendrait à segmenter sur le seul Montant et à isoler une poignée de grossistes. Une transformation `log1p` ramène la skewness à 0,27 et 1,00, puis une standardisation donne un poids égal aux trois dimensions. Après transformation, Fréquence et Montant restent corrélées à **0,85**. Cette redondance est assumée, une PCA (PC1 = 76 % de la variance, axe « valeur client ») confirme une structure essentiellement bidimensionnelle, mais on conserve R, F, M bruts pour garder des centroïdes directement interprétables par le marketing. Deux variantes (panier moyen, RFM net des retours) sont préparées pour tester la robustesse en Partie 3.

**Questions du rapport traitées ici** : Q2 (log + standardisation) ✅ · Q3 (redondance F/M) ✅
